# Figure 4.7: paths of both planners

In [ ]:
from pathlib import Path
import csv
import json
import math
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

BASE = str(Path.home() / 'greenhouse_ws')
WORLD = 'greenhouse_straight_cluttered_s03'

goal = json.load(open(f'{BASE}/worlds/generated/{WORLD}.json'))['goal_pose']
GX, GY = goal['x'], goal['y']

def read_path(file):
    xs, ys = [], []
    for row in csv.DictReader(open(file)):
        try:
            x, y = float(row['tx']), float(row['ty'])
        except ValueError:
            continue
        if math.isfinite(x) and math.isfinite(y):
            xs.append(x)
            ys.append(y)
    return xs, ys

def draw_world(ax):
    root = ET.parse(f'{BASE}/worlds/generated/{WORLD}.sdf').getroot()
    for m in root.find('world').findall('model'):
        name = m.get('name')
        if name == 'ground_plane' or name.startswith('wall'):
            continue
        x, y = (float(v) for v in m.find('pose').text.split()[:2])
        geom = m.find('link/collision/geometry')
        if name.startswith('plant'):
            color = '#5aa05a'
        elif name.startswith('obstacle'):
            color = '#4d4c48'
        else:
            color = '#c9c7bf'
        if geom.find('box') is not None:
            sx, sy, _ = (float(v) for v in geom.find('box/size').text.split())
            ax.add_patch(Rectangle((x - sx / 2, y - sy / 2), sx, sy, color=color))
        else:
            ax.add_patch(Circle((x, y), float(geom.find('cylinder/radius').text), color=color))

runs = [
    ('PPO', f'{BASE}/runs/trajectories_ppo/{WORLD}.csv', '#2a78d6'),
    ('DWB', f'{BASE}/runs/trajectories_dwb_v2/{WORLD}.csv', '#eb6834'),
]

fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)
for ax, (name, file, color) in zip(axes, runs):
    draw_world(ax)
    xs, ys = read_path(file)
    ax.plot(xs, ys, color=color, linewidth=2)
    ax.plot(xs[0], ys[0], 'o', color=color, markersize=8)
    ax.plot(xs[-1], ys[-1], 'X', color='black', markersize=11)
    ax.plot(GX, GY, '*', color='gold', markeredgecolor='black', markersize=18)
    closest = min(math.hypot(GX - x, GY - y) for x, y in zip(xs, ys))
    if closest < 0.75:
        ax.set_title(f'{name}: reached the goal', loc='left')
    else:
        ax.set_title(f'{name}: closest point {closest:.1f} m from the goal', loc='left')
    ax.set_xlim(-0.5, 12.8)
    ax.set_ylim(-1.8, 1.8)
    ax.set_aspect('equal')
    ax.set_ylabel('y (m)')
axes[1].set_xlabel('x (m)')
plt.tight_layout()
plt.savefig('fig4_7_straight_cluttered_s03.png', dpi=300)